<a href="https://colab.research.google.com/github/maliah1010/OpenVocal/blob/experiments%2Fmalia/02_Text_Normalization_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02: Text Normalization Front-End

**Authors:** Malia Hosseini
**Date:** March 2026

### Objective
To build a robust text preprocessing pipeline. Raw text often contains numbers, symbols, and abbreviations that cause Text-to-Speech models to mispronounce or skip words. This notebook creates a sanitization function to convert raw text into fully expanded spoken English prior to phonemization.

## Step 1: Install Front-End Dependencies
To make our engine robust, it needs to understand how to read numbers and symbols aloud. We are installing `inflect`, which is a powerful Python library designed specifically to generate words from numbers (e.g., converting "150" into "one hundred and fifty").

In [1]:
# CELL 1: INSTALL DEPENDENCIES
!pip install -q inflect
print("Dependencies installed!")

Dependencies installed!


## Step 2: Define the Normalization Rules
Here we build the core `normalize_text` function. TTS engines often crash or mispronounce non-alphabetic characters. This function acts as a filter *before* the text reaches the AI, using Regular Expressions (regex) to:
1. Identify and expand currency symbols.
2. Intercept digits and route them through the `inflect` engine to convert them to spoken words.
3. Map common abbreviations (like "Dr." or "St.") to their fully spelled-out forms.

In [2]:
# CELL 2: THE NORMALIZER FUNCTION
import re
import inflect

# Initialize the number-to-words engine
p = inflect.engine()

def normalize_text(text):
    # 1. Handle currency (e.g., $150 -> 150 dollars)
    text = re.sub(r'\$(\d+(?:\.\d{2})?)', r'\1 dollars', text)

    # 2. Expand numbers to words (e.g., 180 -> one hundred and eighty)
    def replace_number(match):
        return p.number_to_words(match.group(0))
    text = re.sub(r'\b\d+\b', replace_number, text)

    # 3. Expand common abbreviations
    abbreviations = {
        r'\bDr\.\b': 'Doctor',
        r'\bMr\.\b': 'Mister',
        r'\bMrs\.\b': 'Missus',
        r'\bSt\.\b': 'Street' # Note: Context matters! This could also be 'Saint'
    }
    for abbr, expansion in abbreviations.items():
        text = re.sub(abbr, expansion, text)

    # 4. Clean up any accidental double spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("Normalization function loaded and ready!")

Normalization function loaded and ready!


## Step 3: The Front-End Stress Test
Here we test our `normalize_text` function against a "messy" input string. If successful, it will expand the currency, convert the digits into spoken words, expand the titles, and clean up the spacing, resulting in a clean phonetic string ready for the TTS engine.

In [3]:
# CELL 3: RUN THE STRESS TEST

# 1. Define a terribly formatted sentence
messy_text = "Dr. Smith owes Mr. Jones $150 for the 2 items on Main St."

# 2. Run it through our pipeline
cleaned_text = normalize_text(messy_text)

# 3. Print the results to compare!
print("ORIGINAL TEXT:")
print(messy_text)
print("\n-------------------\n")
print("NORMALIZED TEXT (Ready for TTS):")
print(cleaned_text)

ORIGINAL TEXT:
Dr. Smith owes Mr. Jones $150 for the 2 items on Main St.

-------------------

NORMALIZED TEXT (Ready for TTS):
Dr. Smith owes Mr. Jones one hundred and fifty dollars for the two items on Main St.
